In [4]:
!pip install -q sentence-transformers datasets torch

In [5]:
## Preparing Training Data (Anchor-Positive Pairs)
## Define domain-specific training pairs (for example, matching
## customer questions to technical solution documents).

from sentence_transformers import InputExample
from torch.utils.data import DataLoader

# Create sample training pairs (Anchor, Positive)
train_data = [
    InputExample(texts=["How do I reset my password?", "Go to settings and click reset password."]),
    InputExample(texts=["Where can I view my monthly invoice?", "Navigate to billing statements in your account portal."]),
    InputExample(texts=["How to increase API rate limits?", "Contact customer support or upgrade your subscription plan."]),
    InputExample(texts=["What is the refund policy?", "Full refunds are available within 14 days of purchase."]),
    InputExample(texts=["How do I enable 2FA authentication?", "Turn on multi-factor authentication under security settings."]),
]

# Wrap data in PyTorch DataLoader
train_dataloader = DataLoader(train_data, shuffle=True, batch_size=2)

In [6]:
## Fine-Tuning with Contrastive Loss
## (MultipleNegativesRankingLoss)
## Load a lightweight base model and fine-tune it on your custom domain data.
from sentence_transformers import SentenceTransformer, losses

# 1. Load base representation model
model = SentenceTransformer("distilbert-base-uncased")

# 2. Define Contrastive Loss
# MultipleNegativesRankingLoss uses other items in the batch as negative examples
train_loss = losses.MultipleNegativesRankingLoss(model)

# 3. Fine-Tune Model
print("--- STARTING CONTRASTIVE FINE-TUNING ---")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=2,
    show_progress_bar=True
)
print("Fine-tuning completed successfully!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

--- STARTING CONTRASTIVE FINE-TUNING ---


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Fine-tuning completed successfully!


In [7]:
## Evaluating Before vs. After Embedding Alignment
## Compare the cosine similarity scores of your custom fine-tuned model against a baseline query.
from sentence_transformers import util

query = "Where are my billing invoices?"
doc_1 = "Navigate to billing statements in your account portal." # Highly relevant
doc_2 = "How to increase API rate limits?"                        # Irrelevant

# Encode texts
q_emb = model.encode(query, convert_to_tensor=True)
d1_emb = model.encode(doc_1, convert_to_tensor=True)
d2_emb = model.encode(doc_2, convert_to_tensor=True)

sim_d1 = util.cos_sim(q_emb, d1_emb).item()
sim_d2 = util.cos_sim(q_emb, d2_emb).item()

print("--- EVALUATION RESULTS ---")
print(f"Query: '{query}'")
print(f"Similarity with Relevant Doc:   {sim_d1:.4f}")
print(f"Similarity with Irrelevant Doc: {sim_d2:.4f}")

--- EVALUATION RESULTS ---
Query: 'Where are my billing invoices?'
Similarity with Relevant Doc:   0.7300
Similarity with Irrelevant Doc: 0.7153
